# Importar bibliotecas

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType

# Leitura da tabela em bronze

In [0]:
df = spark.table('workspace.bronze.erp_px_cat_g1v2')

# Transformações para camada prata

## Limpeza de espaços em branco

In [0]:
# df.limit(5).display()
for i in df.schema.fields:
    if isinstance(i.dataType, StringType):
        df = df.withColumn(i.name, F.trim(F.col(i.name)))

## Normalização do flag de manutenção

In [0]:
df = df.withColumn(
    'MAINTENANCE',
    F.when(F.upper(F.col('MAINTENANCE')) == 'YES', F.lit(True))
     .when(F.upper(F.col('MAINTENANCE')) == 'NO', F.lit(False))
     .otherwise(None)
) # ).select('MAINTENANCE').distinct().display()

## Renomeando colunas

In [0]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Escrita na tabela prata

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_product_category')